# Long-prefix speedup validation
This is a separate workload from the short-prefix experiment, which did not establish a reliable speedup. It tests 1,024-token prefixes at 256 MiB on NVIDIA T4. Select GPU and run all.

One group = 60 requests, 3 repeats, no-cache/LRU-shared/learned-shared plus an independent correctness oracle: 10 runs / 600 request executions. Run the predefined seeds 101, 102, 103 in separate groups and download each ZIP.

Primary measures: paired mean HTTP latency and throughput. p95 is also reported and may regress because cache misses still need full prefill. All policies use the same inputs, precision and final-token-only logits. Profiling disabled. No result is assumed.

Upload gpu-serving-source.zip when prompted. No Drive or GitHub connectivity is required for the project source.


In [ ]:
import os, sys, subprocess, zipfile, hashlib, json
from pathlib import Path
from google.colab import files
uploaded = files.upload()  # Select gpu-serving-source.zip
repo = Path("/content/kv-optimization-upload")
repo.mkdir(exist_ok=True)
source = next((name for name in uploaded if name.endswith(".zip")), None)
assert source, "Upload gpu-serving-source.zip first"
with zipfile.ZipFile(source) as z:
    assert "source-manifest.json" in z.namelist(), "This is not the source bundle"
    for name in z.namelist():
        assert (repo / name).resolve().is_relative_to(repo.resolve()), "Invalid archive path"
    z.extractall(repo)
for name, digest in json.loads((repo / "source-manifest.json").read_text()).items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == digest
os.chdir(repo)
os.environ["USE_TF"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[serving]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select GPU runtime'; print(torch.cuda.get_device_name())"], check=True)
subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([python, "-m", "kv_cache.cli", "experiment", "--output", "results/experiment"], check=True)


## Run one seed
Start with SEED=101. After downloading the result, rerun this cell with 102 and then 103. Keep every other setting unchanged, including the predictor. Do not select only favorable seeds. The predictor was trained on shorter prefixes; this explicitly tests out-of-distribution length.

A descriptive consistency check is at least 5% mean-latency reduction and throughput gain in every paired repeat across all three seeds. Meeting this small synthetic-study threshold is not proof of general production speedup. Failed timing thresholds remain valid results; output mismatches stop the run.


In [ ]:
CONCURRENCY = 1
WORKLOAD = "hot"
SEED = 101
REQUESTS = 60
REPEATS = 3
BUDGET_MIB = 256
PREFIX_TOKENS = 1024
from datetime import datetime, timezone
run_dir = Path("/content") / ("kv-long-prefix-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
archive = str(run_dir) + ".zip"
command = [python, "-u", "-m", "kv_cache.serve_eval", "--predictor", "results/experiment/predictor.json",
           "--device", "cuda", "--requests", str(REQUESTS), "--repeats", str(REPEATS),
           "--seed", str(SEED), "--workload", WORKLOAD, "--budget-mib", str(BUDGET_MIB),
           "--concurrency", str(CONCURRENCY), "--prefix-tokens", str(PREFIX_TOKENS),
           "--no-profile-stages", "--logits-mode", "last", "--reference-full-logits",
           "--policies", "none", "lru", "learned", "--copy-modes", "shared",
           "--output", str(run_dir)]
try:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Evaluation failed; download diagnostics below")
    subprocess.run([python, "-m", "kv_cache.serve_summary", str(run_dir)], check=True)
finally:
    if run_dir.exists():
        import shutil
        shutil.copyfile(repo / "source-manifest.json", run_dir / "source-manifest.json")
        shutil.make_archive(str(run_dir), "zip", run_dir)
    if Path(archive).exists():
        files.download(archive)


In [ ]:
manifest = json.loads((run_dir / "manifest.json").read_text())
print("Correctness:", manifest["status"], "Seed:", SEED)
comparison = json.loads((run_dir / "comparison.json").read_text())
for row in comparison["groups"]:
    print(row["policy"], row["copy_mode"],
          "mean-latency reduction per repeat %:", row["paired_mean_latency_reduction_percent"],
          "throughput gain per repeat %:", row["paired_throughput_gain_percent"],
          "p95 reduction per repeat %:", row["paired_p95_reduction_percent"])
    if row["policy"] != "none":
        meets = all(v >= 5 for v in row["paired_mean_latency_reduction_percent"]) and all(v >= 5 for v in row["paired_throughput_gain_percent"])
        print("Descriptive 5% threshold met for this seed:", meets)
print("Keep results for seeds 101, 102 and 103; do not infer general speedup from one seed.")


## Download again if needed
This cell only downloads the most recent ZIP. It does not rerun the experiment. If the runtime was reset, an undownloaded archive may no longer exist.


In [ ]:
from pathlib import Path
from google.colab import files
archives = sorted(Path("/content").glob("kv-long-prefix-*.zip"))
if archives:
    files.download(str(archives[-1]))
else:
    print("No serving archive found in this runtime.")
